# import

In [1]:
import pandas as pd
import json
import os

# Créer le dossier processed s'il n'existe pas
os.makedirs("../dataset/processed", exist_ok=True)

print("✅ Importations OK")

✅ Importations OK


# 2: Charger le CSV nettoyé

In [2]:
df = pd.read_csv("../dataset/processed/resume_jd_match_cleaned.csv")
print(f"Chargé: {len(df)} exemples")
print("\nDistribution des labels avant équilibrage:")
print(df["label"].value_counts())

Chargé: 6240 exemples

Distribution des labels avant équilibrage:
label
No Fit           3142
Potential Fit    1556
Good Fit         1542
Name: count, dtype: int64


# 3: Équilibrer les labels

In [3]:
min_count = df["label"].value_counts().min()
print(f"Nombre minimum par label: {min_count}")

df_balanced = df.groupby("label").head(min_count)

print("\nDistribution des labels après équilibrage:")
print(df_balanced["label"].value_counts())
print(f"\nTotal après équilibrage: {len(df_balanced)} exemples")

Nombre minimum par label: 1542

Distribution des labels après équilibrage:
label
No Fit           1542
Potential Fit    1542
Good Fit         1542
Name: count, dtype: int64

Total après équilibrage: 4626 exemples


# 4: Fonction de conversion label -> analyse détaillée

In [4]:
def extract_skills_from_job_description(jd_text):
    """Extrait les compétences clés d'une offre d'emploi"""
    jd_lower = jd_text.lower()
    common_skills = ["python", "java", "sql", "aws", "docker", "kubernetes", 
                     "javascript", "react", "angular", "machine learning", "tensorflow",
                     "pytorch", "c++", "c#", "excel", "power bi", "tableau",
                     "project management", "agile", "scrum", "leadership"]
    
    found = [skill for skill in common_skills if skill in jd_lower]
    return found[:5]  # Max 5 compétences

def generate_custom_analysis(cv_text, job_description, label):
    """Génère une analyse PERSONNALISÉE basée sur le contenu réel"""
    
    cv_lower = cv_text.lower()
    
    # 1. Extraire les compétences de l'offre
    required_skills = extract_skills_from_job_description(job_description)
    
    # 2. Vérifier quelles compétences sont dans le CV
    skills_found = []
    skills_missing = []
    
    for skill in required_skills:
        if skill in cv_lower:
            skills_found.append(skill)
        else:
            skills_missing.append(skill)
    
    # 3. Calculer un score PERSONNALISÉ
    if label == "Good Fit":
        base_score = 75
        if len(required_skills) > 0:
            match_rate = len(skills_found) / len(required_skills)
            score = int(base_score + (match_rate * 20))
        else:
            score = 85
    elif label == "Potential Fit":
        base_score = 50
        if len(required_skills) > 0:
            match_rate = len(skills_found) / len(required_skills)
            score = int(base_score + (match_rate * 25))
        else:
            score = 60
    else:  # No Fit
        base_score = 10
        if len(required_skills) > 0:
            match_rate = len(skills_found) / len(required_skills)
            score = int(base_score + (match_rate * 30))
        else:
            score = 25
    
    # 4. Générer des points forts PERSONNALISÉS
    if skills_found:
        points_forts = ", ".join(skills_found[:3])
    else:
        if label == "Good Fit":
            points_forts = "Profil correspondant aux attentes du poste"
        elif label == "Potential Fit":
            points_forts = "Base solide, potentiel de développement"
        else:
            points_forts = "Points transférables limités"
    
    # 5. Générer des points faibles PERSONNALISÉS
    if skills_missing:
        points_faibles = f"Manque: {', '.join(skills_missing[:3])}"
    else:
        if label == "Good Fit":
            points_faibles = "À confirmer en entretien"
        elif label == "Potential Fit":
            points_faibles = "Certaines compétences clés à développer"
        else:
            points_faibles = "Profil non adapté au poste"
    
    # 6. Générer un verdict PERSONNALISÉ
    if score >= 75:
        verdict = "Bon candidat, recommandé pour entretien"
    elif score >= 50:
        verdict = "Candidat potentiel, nécessite formation complémentaire"
    else:
        verdict = "Candidat non recommandé pour ce poste"
    
    return f"""SCORE: {score}%

POINTS FORTS:
- {points_forts}

POINTS FAIBLES:
- {points_faibles}

VERDICT:
{verdict}"""

# 5: Transformer en format conversationnel JSONL

In [5]:
output_file = "../dataset/processed/conversations.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for idx, row in df_balanced.iterrows():
        conv = {
            "messages": [
                {
                    "role": "system",
                    "content": "Tu es un expert RH spécialisé dans l'analyse de CV. Pour chaque offre d'emploi et CV, tu dois fournir un score (0-100), les points forts, les points faibles, et un verdict."
                },
                {
                    "role": "user",
                    "content": row["text_clean"] #les 2000 premiers caractères
                },
                {
                    "role": "assistant",
                    "content": generate_custom_analysis(row["text_clean"], row["text_clean"], row["label"]) #call function to convert label to detailed analysis
                }
            ]
        }
        f.write(json.dumps(conv, ensure_ascii=False) + "\n")

print(f"✅ Sauvegardé: {len(df_balanced)} exemples")
print(f"📁 Fichier: {output_file}")

✅ Sauvegardé: 4626 exemples
📁 Fichier: ../dataset/processed/conversations.jsonl


# 6: Vérifier le premier exemple

In [6]:
with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    first = json.loads(f.readline())

print("=" * 50)
print("SYSTEM:")
print(first["messages"][0]["content"][:200])
print("\n" + "=" * 50)
print("USER (début):")
print(first["messages"][1]["content"][:300])
print("\n" + "=" * 50)
print("ASSISTANT:")
print(first["messages"][2]["content"])
print("\n" + "=" * 50)
print("✅ Format JSONL valide")

SYSTEM:
Tu es un expert RH spécialisé dans l'analyse de CV. Pour chaque offre d'emploi et CV, tu dois fournir un score (0-100), les points forts, les points faibles, et un verdict.

USER (début):
For the given job description <<Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for our accelerated growth of 300% in the last 3 years with over 5500+ employees globally, with over 30+ locations in the US and global operations in 32 coun

ASSISTANT:
SCORE: 40%

POINTS FORTS:
- sql, excel, project management

POINTS FAIBLES:
- Profil non adapté au poste

VERDICT:
Candidat non recommandé pour ce poste

✅ Format JSONL valide


# 7: Statistiques finales

In [7]:
print("=" * 50)
print("📊 RÉSUMÉ FINAL DU DATASET")
print("=" * 50)
print(f"Total exemples: {len(df_balanced)}")
print(f"\nDistribution:")
print(df_balanced["label"].value_counts())
print(f"\nFichier de sortie: ../dataset/processed/conversations.jsonl")
print("✅ Prêt pour le fine-tuning QLoRA !")

📊 RÉSUMÉ FINAL DU DATASET
Total exemples: 4626

Distribution:
label
No Fit           1542
Potential Fit    1542
Good Fit         1542
Name: count, dtype: int64

Fichier de sortie: ../dataset/processed/conversations.jsonl
✅ Prêt pour le fine-tuning QLoRA !
